# 08.3 在 NSynth 子集上训练 toy WaveNet

本 Notebook 调用 `wavenet/train.py` 的真实训练入口。默认配置用于展示完整工程路径：数据检查、训练、checkpoint、loss 表、逐采样生成和音频保存。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, display

from _common.audio_io import save_audio
from _common.checkpointing import load_torch_checkpoint
from _common.dataset_registry import check_required_assets
from _common.device_utils import choose_device
from _common.paths import portable_path
from _common.plotting import finish_figure, plot_spectrogram, setup_plot_style
from wavenet.dataset import AudioWindowConfig, ManifestWaveformDataset
from wavenet.generate import sample_tokens, tokens_to_audio
from wavenet.model import build_wavenet_from_config
from wavenet.train import train_wavenet

OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_AUDIO = ROOT / "output_audio" / "08_3"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_CKPT = ROOT / "outputs" / "checkpoints" / "wavenet_08_3"
NSYNTH_MANIFEST = ROOT / "data_manifests" / "nsynth_subset.csv"
for path in [OUTPUT_FIGURES, OUTPUT_AUDIO, OUTPUT_TABLES, OUTPUT_CKPT]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()
DEVICE = choose_device(os.getenv("CHAPTER08_DEVICE", "auto"))
print("selected device:", DEVICE)

def rel(path):
    return portable_path(path, ROOT)


In [ ]:
check_required_assets(
    ["nsynth"],
    message="08_3 requires NSynth JSON/WAV. Download it before running WaveNet training.",
    stop=True,
)
if not NSYNTH_MANIFEST.exists():
    print("Missing NSynth subset manifest:", rel(NSYNTH_MANIFEST))
    print("Build it with: python data_manifests/build_chapter08_manifests.py --nsynth-limit 256")
    raise FileNotFoundError(NSYNTH_MANIFEST)
print("Using NSynth manifest:", rel(NSYNTH_MANIFEST))


In [ ]:
config = {
    "seed": 8,
    "device": DEVICE,
    "data": {
        "source": "nsynth",
        "manifest_csv": rel(NSYNTH_MANIFEST),
        "split": "valid",
        "max_files": 24,
        "sample_rate": 16000,
        "window_samples": 1024,
        "quantization_channels": 64,
    },
    "model": {
        "residual_channels": 16,
        "dilation_channels": 16,
        "skip_channels": 32,
        "kernel_size": 2,
        "layers_per_cycle": 4,
        "dilation_cycles": 1,
    },
    "training": {
        "batch_size": 4,
        "epochs": 3,
        "learning_rate": 0.001,
        "max_steps_per_epoch": 4,
    },
    "outputs": {
        "checkpoint_dir": rel(OUTPUT_CKPT),
        "history_csv": rel(OUTPUT_TABLES / "08_3_wavenet_history.csv"),
    },
}
config


In [ ]:
history = train_wavenet(config)
history_df = pd.DataFrame(history)
display(history_df)


**简化 WaveNet 训练损失曲线**

训练只跑三个轮次、每轮 4 步，合计 12 次参数更新，交叉熵只能缓慢下降。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", color="0.15")
ax.set_xlabel("轮次")
ax.set_ylabel("交叉熵")
finish_figure(fig, OUTPUT_FIGURES / "08_3_wavenet_training_loss.png")
plt.show()


下面从一段真实音频中取 64 个 token 作为种子，让模型逐点预测 1024 个新采样点。在 16 kHz 采样率下，种子与新生成的部分合计约 0.07 秒，下方播放器里只能听到一声短促的响动，而不是一个完整的音符。

听感接近噪声同样在意料之中。这个简化模型的感受野只有 16 个采样点（约 1 毫秒），训练也只有几十次参数更新，学到的仅是信号最粗浅的局部纹理。原因在于，此处的主要目标是走通完整工程路径：数据检查、训练、checkpoint、逐采样生成与音频保存。要逼近 WaveNet 论文中的效果，需要深得多的膨胀卷积堆叠、更长的上下文窗口和充分得多的训练。这也是后续章节转向神经音频编解码器的动机之一。


In [ ]:
checkpoint = load_torch_checkpoint(OUTPUT_CKPT / "last.pt", map_location="cpu")
model = build_wavenet_from_config(config)
model.load_state_dict(checkpoint["model"])
model.to(DEVICE)
model.eval()

dataset = ManifestWaveformDataset(
    manifest_csv=config["data"]["manifest_csv"],
    max_files=1,
    config=AudioWindowConfig(
        sample_rate=config["data"]["sample_rate"],
        window_samples=config["data"]["window_samples"],
        quantization_channels=config["data"]["quantization_channels"],
    ),
)
seed_tokens, _ = dataset[0]
generated_tokens = sample_tokens(
    model,
    seed_tokens[:64],
    num_new_tokens=1024,
    temperature=0.95,
    device=DEVICE,
)
generated_audio = tokens_to_audio(
    generated_tokens,
    quantization_channels=config["data"]["quantization_channels"],
)
save_audio(OUTPUT_AUDIO / "wavenet_generated.wav", generated_audio, config["data"]["sample_rate"])
print("generated token shape:", tuple(generated_tokens.shape))
display(Audio(str(OUTPUT_AUDIO / "wavenet_generated.wav")))


**简化 WaveNet 生成音频的频谱图**

频谱上看不到稳定的谐波结构，能量在各频段快速抖动，与播放器里噪声般的听感一致。


In [ ]:
fig = plot_spectrogram(
    generated_audio,
    config["data"]["sample_rate"],
    title=None,
    out_path=OUTPUT_FIGURES / "08_3_wavenet_generated_spectrogram.png",
    n_fft=512,
    hop_length=128,
)
plt.show()

print("08_3 outputs:")
for path in [
    OUTPUT_TABLES / "08_3_wavenet_history.csv",
    OUTPUT_CKPT / "last.pt",
    OUTPUT_AUDIO / "wavenet_generated.wav",
    OUTPUT_FIGURES / "08_3_wavenet_training_loss.png",
    OUTPUT_FIGURES / "08_3_wavenet_generated_spectrogram.png",
]:
    print("-", rel(path), "exists=", path.exists())
